# Cookie Cats A/B Test: Bootstrap Analysis
Checks whether the retention difference between gate 30 and gate 40 is real or just luck.

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
client = bigquery.Client(project="acoustic-scarab-406920")

df = client.query("""
SELECT * FROM `acoustic-scarab-406920.cookie_cats.players_clean`
""").to_dataframe()

df.head()

## Bootstrap
Resample players 1,000 times and record the retention difference (gate 30 minus gate 40) each time.

In [ ]:
import pandas as pd

def bootstrap(df, column, n=1000):
    diffs = []
    for i in range(n):
        sample = df.sample(frac=1, replace=True, random_state=i)
        rates = sample.groupby('version')[column].mean()
        diffs.append(rates['gate_30'] - rates['gate_40'])
    return pd.Series(diffs)

diffs_7 = bootstrap(df, 'retention_7')
diffs_1 = bootstrap(df, 'retention_1')

print('7 day: share of resamples where gate 30 wins:', (diffs_7 > 0).mean())
print('1 day: share of resamples where gate 30 wins:', (diffs_1 > 0).mean())

## Distribution of differences
Most of the histogram sitting right of 0 means gate 30 is consistently better.

In [ ]:
ax = diffs_7.plot(kind='hist', bins=30, alpha=0.6, label='7 day')
diffs_1.plot(kind='hist', bins=30, alpha=0.6, label='1 day', ax=ax)
ax.axvline(0, color='black')
ax.set_xlabel('Retention difference (gate 30 minus gate 40)')
ax.legend()